In [46]:
%pip install langchain-ollama

from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated, Any
from pathlib import Path

# Use Pydantic v1 (LangChain's default) compatible validators
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
# --- 1. Robust Pydantic Models (The Fix) ---

class Section(BaseModel):
    title: str
    description: str = Field(..., description="What to cover in this section")

    # This validator fixes the keys if DeepSeek renames them
    @root_validator(pre=True)
    def map_deepseek_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        # Map 'section_title' -> 'title'
        if "section_title" in values:
            values["title"] = values.pop("section_title")
        # Map 'section_description' -> 'description'
        if "section_description" in values:
            values["description"] = values.pop("section_description")
        return values

In [48]:
class Plan(BaseModel):
    title: str = Field(..., description="The main title of the blog post")
    sections: List[Section]

    # This validator unwraps the content if DeepSeek nests it in 'blog_plan'
    @root_validator(pre=True)
    def unwrap_deepseek_nesting(cls, values: dict[str, Any]) -> dict[str, Any]:
        # If the model returned {"blog_plan": {...}}, unwrap it
        if "blog_plan" in values:
            return values["blog_plan"]
        return values

In [49]:
# --- State Definition ---
class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[List[str], operator.add] 
    final: str

In [50]:
# 2. Initialize the Ollama LLM
# We use the specific model tag you provided.
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud",
    temperature=0,
    # If your cloud instance is at a custom URL, uncomment and set base_url:
    # base_url="https://your-ollama-cloud-endpoint" 
)

In [51]:
def orchestrator(state: State) -> dict:
    """Generates the plan using structured output."""
    
    structured_llm = llm.with_structured_output(Plan)

    # We add a strict example in the prompt to guide the model
    plan = structured_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic. "
                    "Return ONLY valid JSON. "
                    "Structure: { 'title': 'Blog Title', 'sections': [{ 'title': 'Section Header', 'description': 'Content info' }] }"
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

In [52]:
def fanout(state: State):
    """Creates a parallel execution branch for every section in the plan."""
    return [
        Send("worker", {
            "section": section, 
            "topic": state["topic"], 
            "blog_title": state["plan"].title
        }) 
        for section in state["plan"].sections
    ]

In [53]:
def worker(payload: dict) -> dict:
    """Processes a single section (Runs in parallel)."""
    
    section: Section = payload["section"]
    topic = payload["topic"]
    blog_title = payload["blog_title"]

    response = llm.invoke(
        [
            SystemMessage(content="You are a technical blog writer."),
            HumanMessage(
                content=(
                    f"Blog Title: {blog_title}\n"
                    f"Main Topic: {topic}\n"
                    f"Section Title: {section.title}\n"
                    f"Section Goal: {section.description}\n\n"
                    "Write this section in clean Markdown. Do not include the title header."
                )
            ),
        ]
    )
    
    return {"sections": [f"## {section.title}\n\n{response.content}"]}

In [54]:
from pathlib import Path

def reducer(state: State) -> dict:
    """Combines all sections into the final file."""
    
    title = state["plan"].title
    
    # Sort or just join (LangGraph executes in parallel, so order might vary slightly without an ID)
    body = "\n\n".join(state["sections"]).strip()
    final_md = f"# {title}\n\n{body}\n"

    # Save to file (Sanitize filename)
    safe_title = "".join([c if c.isalnum() else "_" for c in title])
    filename = f"{safe_title}.md"
    
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")
    
    print(f"--> Blog post saved to: {filename}")
    return {"final": final_md}

In [55]:
# --- Graph Construction ---

g = StateGraph(State)

g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

In [57]:
# --- Execution ---

if __name__ == "__main__":
    print("Starting generation with DeepSeek Cloud...")
    
    inputs = {
        "topic": "I Installed 5 Productivity Apps and Still Did Nothing", 
        "sections": [] 
    }
    
    try:
        output = app.invoke(inputs)
        print("Success! Check the folder for the .md file.")
    except Exception as e:
        print(f"Error occurred: {e}")

Starting generation with DeepSeek Cloud...
--> Blog post saved to: I_Installed_5_Productivity_Apps_and_Still_Did_Nothing.md
Success! Check the folder for the .md file.
